# Python 5: classes, dataclasses and enums

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-05-classes-and-dataclasses.ipynb)

Up to now you have used the values Python gives you: numbers, strings, lists and dictionaries. A **class** lets you define your own kind of value, with its own data and its own functions. Pipelines use classes for connections, settings, records and pluggable steps. This chapter starts from what a class is and how to write one. Then it covers inheritance, abstract base classes, dataclasses for records, and enums for fixed sets of values.

Every rule is stated before the code that shows it, and each code cell's `assert` lines are the expected results. If an `assert` fails, your model of that rule is wrong.

## What is a class?

A **class** is a blueprint for a kind of object. An **object**, also called an **instance**, is one value built from that blueprint. Each object has its own data, called **attributes**, and can run functions defined in the class, called **methods**.

A `dict` is a useful comparison. `{"name": "Ana"}` holds data, but it has no rule about which keys it must contain and no behaviour attached to it. A class keeps the data, the rules and the behaviour together.

You already use classes. `[1, 2, 3]` is an instance of `list`, and `type([1, 2, 3])` shows it.

### Where you'll use it

- **Connectors and clients:** a database client object holds its host, credentials and retry settings, and its methods run the queries.
- **Pipeline steps:** a `Transform` object holds its configuration and exposes one `run(batch)` method.

In [ ]:
numbers = [1, 2, 3]

assert type(numbers) is list          # numbers is an instance of the class list
assert isinstance(numbers, list)
assert numbers.append is not None     # append is a method that every list has

## Your first class

A class is defined with the `class` keyword, a name in `CamelCase`, and a colon. The body is indented. `pass` is a statement that does nothing, which lets an empty body be valid.

Creating an object is called **instantiation**. You call the class as if it were a function, and the result is a new instance. Two instances are two separate objects, even when they come from the same class.

In [ ]:
class Customer:
    pass

ana = Customer()                 # create an instance
ben = Customer()                 # a second, separate instance

assert type(ana) is Customer
assert isinstance(ana, Customer)
assert ana is not ben            # different objects, from the same blueprint

## The constructor: `__init__` sets up the data

`__init__` is the **constructor**. Python calls it right after it creates an object, and uses it to set the starting attributes. Its first parameter is always `self`, which stands for the object being built. Inside the class, `self.name = name` stores a value on that object.

Each instance has its own attributes. Changing one customer does not change another.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name             # store the value on this object
        self.city = city

ana = Customer("Ana Lopez", "Oslo")
ben = Customer("Ben Smith", "Lima")

assert ana.name == "Ana Lopez"       # read an attribute with a dot
assert ben.city == "Lima"

ana.city = "Madrid"                  # change one object's attribute
assert ana.city == "Madrid"
assert ben.city == "Lima"            # the other object is untouched

## Methods: functions that belong to the object

A **method** is a function defined inside the class. It always receives the object as its first argument, `self`, so it can read and change the object's attributes. You call it with a dot: `ana.greeting()`. Python passes `ana` in as `self` for you.

`ana.greeting()` is shorthand for `Customer.greeting(ana)`. Both calls run the same function on the same object.

### Where you'll use it

- **Domain behaviour next to data:** an `Invoice` object that knows how to compute its own total and tax, so the rule lives in one place.
- **Testability:** a method on a small object is tested by creating the object and calling the method, with no network or database.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city
        self.orders = []

    def greeting(self):
        return f"Hello, {self.name} from {self.city}"

    def add_order(self, amount):
        self.orders.append(amount)

    def total_spent(self):
        return sum(self.orders)

ana = Customer("Ana Lopez", "Oslo")
assert ana.greeting() == "Hello, Ana Lopez from Oslo"
assert Customer.greeting(ana) == ana.greeting()      # the same call, written out

ana.add_order(120)
ana.add_order(45.5)
assert ana.total_spent() == 165.5
assert Customer("Ben", "Lima").total_spent() == 0    # a new object starts with no orders

## Printing an object: `__repr__` and `__str__`

By default, printing an object shows only its type and memory address, which tells you nothing. Two special methods control the text:

- `__repr__` returns the **developer** text, shown in logs and in the notebook. Write it so the log line says what the object holds.
- `__str__` returns the **user** text, used by `print` and by `f"{obj}"`. If you do not define it, Python falls back to `__repr__`.

Methods with two underscores on each side are **dunder methods**. Python calls them for you in certain situations, such as `print`, `==` and `len`.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city

    def __repr__(self):
        return f"Customer(name={self.name!r}, city={self.city!r})"

ana = Customer("Ana Lopez", "Oslo")
assert repr(ana) == "Customer(name='Ana Lopez', city='Oslo')"
assert str(ana) == repr(ana)                       # no __str__, so str() uses __repr__

class Tag:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Tag({self.name!r})"

    def __str__(self):
        return f"#{self.name}"

tag = Tag("etl")
assert str(tag) == "#etl" and repr(tag) == "Tag('etl')"
assert f"{tag}" == "#etl"

## Equality and hashing

By default, `==` on two objects is `True` only when they are the **same** object. For a record such as an account, you usually want "same number means equal". Define `__eq__` to say so.

Defining `__eq__` has a side effect: Python sets `__hash__` to `None`, so the objects can no longer be dictionary keys or set members. A key must have a stable hash, so define `__hash__` as well. Return `NotImplemented` from `__eq__` for other types, so Python can try the reverse comparison before it answers `False`.

In [ ]:
class Account:
    def __init__(self, number, owner):
        self.number = number
        self.owner = owner

    def __eq__(self, other):
        if not isinstance(other, Account):
            return NotImplemented
        return self.number == other.number

    def __hash__(self):
        return hash(self.number)          # equal objects must have equal hashes

assert Account("1", "Ana") == Account("1", "Ben")          # equal by number
assert Account("1", "Ana") != Account("2", "Ana")
assert Account("1", "Ana") is not Account("1", "Ana")      # still two objects
assert {Account("1", "Ana"): "open"}[Account("1", "Ben")] == "open"
assert Account("1", "Ana") != "1"                          # other types are never equal

class NoHash:
    def __eq__(self, other):
        return True

assert NoHash.__hash__ is None                             # defining __eq__ removed __hash__
try:
    {NoHash(): 1}
except TypeError:
    pass
else:
    raise AssertionError("an unhashable object cannot be a dictionary key")

## Class attributes versus instance attributes

An instance attribute belongs to one object (`self.name`). A **class attribute** is written in the class body, outside any method, and is shared by every instance. Read it through the class or through an instance.

Assigning `self.x` inside a method creates an instance attribute, even when a class attribute with the same name exists. Changing the class attribute later does not change instances that already have their own copy.

### Where you'll use it

- **Default settings:** a class-level default timeout that each instance can override.
- **Counters:** a class attribute that counts how many objects were created, for metrics in a long-running worker.

In [ ]:
class Pipeline:
    default_retries = 3                   # class attribute: one value for the whole class

    def __init__(self, name):
        self.name = name                  # instance attribute: one value per object
        self.retries = Pipeline.default_retries

a, b = Pipeline("a"), Pipeline("b")
assert a.default_retries == 3             # readable through an instance too

Pipeline.default_retries = 5              # change the class attribute
assert a.retries == 3 and b.retries == 3     # objects that already exist keep their copy
assert Pipeline("c").retries == 5            # new objects see the new value

## Properties: run code on read and write

A **property** looks like a plain attribute (`p.days`) but runs code when you read it or assign to it. Use it to keep an object valid: reject a bad value at the moment it is assigned, not much later when the value is used.

- `@property` marks the getter, which runs on `p.days`.
- `@days.setter` marks the setter, which runs on `p.days = ...`.
- A property with no setter is **read-only**. Assigning to it raises `AttributeError`.

### Where you'll use it

- **Guarding invariants:** a partition size, a retry count or a sample rate that must always be valid, rejected at the point it is set.
- **Derived values:** a `row_count` that is computed from the data each time, so it can never go stale.

In [ ]:
class Partition:
    def __init__(self, days):
        self.days = days                  # goes through the setter below

    @property
    def days(self):
        return self._days

    @days.setter
    def days(self, value):
        if value < 1:
            raise ValueError("a partition covers at least one day")
        self._days = value

p = Partition(7)
assert p.days == 7
try:
    p.days = 0
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

class Report:
    def __init__(self, rows):
        self._rows = rows

    @property
    def row_count(self):                  # computed on every read, so never stale
        return len(self._rows)

report = Report([1, 2])
assert report.row_count == 2
try:
    report.row_count = 5                  # no setter: read-only
except AttributeError:
    pass
else:
    raise AssertionError("expected AttributeError")

## Inheritance: reuse a class and change part of it

A class can **inherit** from another class. The new class, the **subclass**, gets every attribute and method of the parent, the **base class**. It can also **override** a method with its own version. Write the parent as the general idea and each child as one specific case.

`super()` calls the parent's version of a method. The child can add to the parent's behaviour instead of copying it.

`isinstance(child, Parent)` is `True`, because a child is also a parent.

### Where you'll use it

- **Pluggable sources and sinks:** a `Source` base class with `CsvSource`, `S3Source` and `ApiSource` subclasses. The pipeline calls `read()` on any of them.
- **Framework hooks:** Airflow operators and many ORM base classes are designed to be subclassed.

In [ ]:
class Source:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"source {self.name}"

class CsvSource(Source):
    def __init__(self, name, path):
        super().__init__(name)            # the parent sets self.name
        self.path = path

    def describe(self):
        return f"{super().describe()} reading {self.path}"

src = CsvSource("orders", "landing/orders.csv")
assert src.name == "orders"                          # inherited from Source
assert src.describe() == "source orders reading landing/orders.csv"
assert isinstance(src, Source)                       # a CsvSource is also a Source
assert issubclass(CsvSource, Source) and not issubclass(Source, CsvSource)

## Abstract base classes: require a method

A base class often describes a method that every child must provide, with no sensible default. An **abstract base class** (ABC) makes that rule enforceable. Inherit from `abc.ABC` and mark the method with `@abstractmethod`. Python then refuses to create an object of any class that still has an abstract method.

### Where you'll use it

- **Plugin contracts:** every source must implement `read()`, and a source that forgets it fails at creation, not in the middle of a run.

In [ ]:
from abc import ABC, abstractmethod

class Reader(ABC):
    @abstractmethod
    def read(self):
        """Return a list of rows."""

class CsvReader(Reader):
    def read(self):
        return ["csv row"]

class ApiReader(Reader):
    def read(self):
        return ["api row"]

class IncompleteReader(Reader):
    pass                                  # forgot to implement read()

def load_all(readers):
    return [row for reader in readers for row in reader.read()]

assert load_all([CsvReader(), ApiReader()]) == ["csv row", "api row"]

for abstract_use in [Reader, IncompleteReader]:
    try:
        abstract_use()
    except TypeError:
        pass
    else:
        raise AssertionError(f"{abstract_use.__name__} should not be instantiable")

## Composition: has-a instead of is-a

Inheritance means "is a". Composition means "has a": an object keeps another object in an attribute and calls its methods. Composition is usually easier to change later, because you can swap the part without touching the whole class.

In [ ]:
class Retry:
    def __init__(self, attempts):
        self.attempts = attempts

class HttpExtractor:
    def __init__(self, url, retry):
        self.url = url
        self.retry = retry                # the extractor has a retry policy

extractor = HttpExtractor("https://api.example.com/orders", Retry(3))
assert extractor.retry.attempts == 3

extractor.retry = Retry(5)                # swap the policy without subclassing
assert extractor.retry.attempts == 5

## Dataclasses: records without the boilerplate

Most classes in a pipeline hold data. Writing `__init__`, `__repr__` and `__eq__` by hand every time is repetitive and easy to get wrong. The `@dataclass` decorator writes those methods for you, from the **field annotations**: lines such as `name: str`, which list the fields and their types.

Below, the same record is written by hand and as a dataclass. Both are checked to behave the same way.

### Where you'll use it

- **Typed records:** an `Event` or `Order` dataclass as the in-memory shape of a row before it is written.
- **Serialisation:** `dataclasses.asdict` turns a record into a dictionary for JSON or for a database insert.

In [ ]:
from dataclasses import dataclass

class OrderByHand:
    def __init__(self, order_id, amount):
        self.order_id = order_id
        self.amount = amount

    def __repr__(self):
        return f"OrderByHand(order_id={self.order_id!r}, amount={self.amount!r})"

    def __eq__(self, other):
        return (self.order_id, self.amount) == (other.order_id, other.amount)

@dataclass
class Order:
    order_id: int
    amount: float

assert OrderByHand(1, 9.5) == OrderByHand(1, 9.5)
assert Order(1, 9.5) == Order(1, 9.5)                    # the same behaviour, far less code
assert repr(Order(1, 9.5)) == "Order(order_id=1, amount=9.5)"

## Dataclass options: validation, ordering and defaults

- `__post_init__` runs after the generated `__init__`. Use it to validate the fields.
- `order=True` generates comparison methods that compare the fields in order, so records can be sorted.
- `field(default_factory=list)` gives each object its own new list. Writing `tags: list = []` would share one list between all objects, which is the mutable-default trap from chapter 3.
- `frozen=True` makes the object **read-only**. A frozen object can be a dictionary key.
- `replace(obj, field=new)` returns a changed copy, and the original stays as it was.
- `asdict(obj)` converts the object to a dictionary.

In [ ]:
from dataclasses import asdict, dataclass, field, replace

@dataclass
class ValidatedOrder:
    order_id: int
    amount: float

    def __post_init__(self):
        if self.amount < 0:
            raise ValueError("amount must not be negative")

assert asdict(ValidatedOrder(1, 9.5)) == {"order_id": 1, "amount": 9.5}
try:
    ValidatedOrder(2, -1.0)
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

@dataclass(order=True)
class Version:
    major: int
    minor: int

assert sorted([Version(1, 2), Version(1, 0), Version(0, 9)]) == [Version(0, 9), Version(1, 0), Version(1, 2)]

@dataclass(frozen=True)
class OrderKey:
    customer_id: int
    order_date: str

@dataclass
class Run:
    job: str
    attempts: int = 0                                  # a default value
    tags: list[str] = field(default_factory=list)      # a new list for every Run

first = Run("load")
second = Run("load")
first.tags.append("nightly")
assert second.tags == []                               # the lists are not shared

key = OrderKey(7, "2024-01-15")
assert key == OrderKey(7, "2024-01-15")
seen = {key: "processed"}                              # frozen objects can be dictionary keys
assert seen[OrderKey(7, "2024-01-15")] == "processed"

updated = replace(first, attempts=1)                   # a changed copy
assert updated.attempts == 1 and first.attempts == 0

try:
    key.customer_id = 8
except AttributeError:                                 # FrozenInstanceError is a subclass of AttributeError
    pass
else:
    raise AssertionError("frozen dataclasses cannot be changed")

## Enums: a fixed set of named values

An **enum** is a class whose objects are a fixed list of named values, such as the statuses a job can have. Each member has a `name` (the label in code) and a `value` (the stored data). Comparing against a member catches typos that a plain string would hide.

A plain `Enum` member never equals its raw value. Inheriting also from `str` makes a member equal to its text. That is useful when a value arrives from a file, and it is the only case where the `str` mixin is worth the extra meaning.

`enum.auto()` assigns values for you, starting at `1`.

### Where you'll use it

- **Status fields:** `OK`, `FAILED` and `SKIPPED` for a job run, so a typo in a status string cannot reach a dashboard.
- **Source systems:** a fixed set of supported formats or regions, checked when a file arrives.

In [ ]:
from enum import Enum, auto

class Status(str, Enum):
    OK = "ok"
    FAILED = "failed"
    SKIPPED = "skipped"

assert Status.OK.name == "OK"
assert Status.OK.value == "ok"
assert Status("failed") is Status.FAILED             # look up a member by its value
assert [s.value for s in Status] == ["ok", "failed", "skipped"]

def is_done(status):
    return status == Status.OK

assert is_done("ok")                                  # the str mixin equals the raw text
assert not is_done("failed")

try:
    Status("ok ")                                     # the trailing space makes a different value
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

class Level(Enum):
    LOW = 1
    HIGH = 2

assert Level.LOW != 1                                 # a plain Enum never equals its raw value
assert Level.LOW.value == 1

class Stage(Enum):
    EXTRACT = auto()
    LOAD = auto()

assert [s.value for s in Stage] == [1, 2]

## Practice questions

1. Write a class `Counter` with an `increment()` method and a `value` property that starts at 0. Create two counters and show that they are independent.
2. Write a frozen dataclass `Money` with `amount` (a `Decimal`) and `currency` (a string), and a method `add(other)` that refuses to add two different currencies.
3. Write an `Enum` called `Region` with `NORTH`, `SOUTH` and `WEST`, and a function that returns the number of members.
4. Write a dataclass `Shipment` with a `weight_kg` field. Reject a weight that is zero or negative when the object is created.

## Answers

### Question 1

In [ ]:
class Counter:
    def __init__(self):
        self._value = 0

    def increment(self):
        self._value += 1

    @property
    def value(self):
        return self._value

a = Counter()
b = Counter()
a.increment()
a.increment()
assert a.value == 2 and b.value == 0

### Question 2

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def add(self, other):
        if self.currency != other.currency:
            raise ValueError(f"cannot add {other.currency} to {self.currency}")
        return Money(self.amount + other.amount, self.currency)

assert Money(Decimal("10.00"), "EUR").add(Money(Decimal("2.50"), "EUR")) == Money(Decimal("12.50"), "EUR")
try:
    Money(Decimal("1"), "EUR").add(Money(Decimal("1"), "USD"))
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

### Question 3

In [ ]:
from enum import Enum

class Region(Enum):
    NORTH = "north"
    SOUTH = "south"
    WEST = "west"

def region_count():
    return len(Region)

assert region_count() == 3
assert Region.WEST.value == "west"

### Question 4

In [ ]:
from dataclasses import dataclass

@dataclass
class Shipment:
    weight_kg: float

    def __post_init__(self):
        if self.weight_kg <= 0:
            raise ValueError("weight must be positive")

assert Shipment(2.5).weight_kg == 2.5
try:
    Shipment(0)
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Interview notes

- **A class is a blueprint, and an object is an instance.** `__init__` sets up each instance, and every method takes `self`.
- **Defining `__eq__` removes `__hash__`.** Define both when objects are dictionary keys or set members.
- **Prefer composition to deep inheritance.** Two levels of subclassing are usually enough. Use an ABC to state the contract a subclass must meet.
- **Dataclasses for records, classes for behaviour.** A dataclass holds data. A class with many methods and hidden state is a service.
- **Never use a mutable default in a dataclass field.** Use `field(default_factory=list)`.
- **Frozen dataclasses give value semantics.** Equal fields mean equal objects, so they are safe as dictionary keys.